# Bài toán 1 — Bảng A (so kiến trúc) + B3 (trần của đề tài), bộ chia khối có đệm

**Dữ liệu:** 3.000 khung nhiệt 640×512, 12.390 hộp OBB nhãn tay, 4 lớp. **Bộ chia:** `chia_khoi.csv`, 5 vòng, đệm 14,3 m, rò rỉ 0,0 %.

**Bốn kiến trúc, thứ tự ưu tiên** (hết GPU thì dừng ở đâu vẫn có bảng):

| # | Model | Hộp | Vai trò |
|---|---|---|---|
| 1 | `yolo26s-obb` | xoay | pipeline dự án — 3 hạt |
| 2 | `yolov9c` | thẳng | kiến trúc Darabi (B1), train lại trên data mình |
| 3 | `yolo26s` | thẳng | cùng kiến trúc dòng 1, tách phần lợi do OBB |
| 4 | `yolov8s-obb` | xoay | model một giai đoạn hộp xoay khác |

**Cấu hình khoá cho mọi dòng** (từ `args.yaml` lượt 09/08): 150 epoch, patience 20, batch 16, imgsz 640, lr0 0,005, degrees 15, flipud/fliplr 0,5, mosaic 1,0, close_mosaic 15.

**Thứ tự chạy:** ô 4 chạy **vòng 0** cho cả 4 kiến trúc (Bảng A chỉ cần một vòng chung) → ô 5 chạy **vòng 1–4** cho `yolo26s-obb` (B3 cần out-of-fold). Mỗi lượt xong là ghi JSON ngay, đứt phiên chạy lại ô 1→3 rồi ô 4/5, lượt nào có JSON thì bỏ qua.

**Trước khi chạy:** `MyDrive/colab_B/` có `khung_nhan.zip` + `chia_khoi.csv` + notebook này. Runtime → GPU (T4 đủ; A100 nhanh gấp ~4).

In [1]:
# 1. Drive + giải nén
from google.colab import drive; drive.mount('/content/drive')
import os, zipfile, json, csv, time, glob, shutil
D = '/content/drive/MyDrive/colab_B'
assert os.path.isdir(D), D
KQ = f'{D}/ket_qua'; os.makedirs(KQ, exist_ok=True)
if not os.path.isdir('/content/khung/images'):
    t = time.time(); zipfile.ZipFile(f'{D}/khung_nhan.zip').extractall('/content/khung'); print('giải nén %.0f s' % (time.time() - t))
rows = list(csv.DictReader(open(f'{D}/chia_khoi.csv', encoding='utf8')))
print(len(rows), 'khung |', len(os.listdir('/content/khung/images')), 'ảnh |', len(os.listdir('/content/khung/labels')), 'nhãn')
!pip -q install ultralytics
import ultralytics; print('ultralytics', ultralytics.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Mounted at /content/drive
giải nén 12 s
3000 khung | 3000 ảnh | 3000 nhãn
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 9.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
ultralytics 8.4.157
Tesla T4, 15360 MiB


In [2]:
# 2. Dựng thư mục theo vai từng vòng (symlink, không chép ảnh) + data.yaml cho OBB và hộp thẳng
LOP = ['SingleHotSpot', 'MultiHotSpot', 'Bypass', 'ModuleOpenCircuit']
K = 5

def obb_sang_hop_thang(src, dst):
    # 8 số 4 góc (chuẩn hoá) -> cx cy w h
    out = []
    for l in open(src, encoding='utf8').read().splitlines():
        q = l.split()
        if len(q) != 9: continue
        xs = [float(q[k]) for k in (1, 3, 5, 7)]; ys = [float(q[k]) for k in (2, 4, 6, 8)]
        x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
        out.append('%s %.6f %.6f %.6f %.6f' % (q[0], (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1))
    open(dst, 'w').write('\n'.join(out) + ('\n' if out else ''))

def dung(vong, kieu):
    # kieu: 'obb' | 'hbb'
    goc = f'/content/ds_{kieu}_v{vong}'
    if os.path.isdir(goc): return goc
    for vai in ('train', 'val', 'test'):
        os.makedirs(f'{goc}/{vai}/images'); os.makedirs(f'{goc}/{vai}/labels')
    for r in rows:
        vai = r[f'vai_v{vong}']
        if vai == 'dem': continue            # đệm: bỏ hẳn
        ten = r['file'][:-4]
        os.symlink(f'/content/khung/images/{r["file"]}', f'{goc}/{vai}/images/{r["file"]}')
        src = f'/content/khung/labels/{ten}.txt'
        dst = f'{goc}/{vai}/labels/{ten}.txt'
        if kieu == 'obb': os.symlink(src, dst)
        else: obb_sang_hop_thang(src, dst)
    open(f'{goc}/data.yaml', 'w').write(
        f'path: {goc}\ntrain: train/images\nval: val/images\ntest: test/images\nnc: 4\nnames: {LOP}\n')
    return goc

for v in range(K):
    for kieu in ('obb', 'hbb'):
        g = dung(v, kieu)
    n = {vai: len(os.listdir(f'{g}/{vai}/images')) for vai in ('train', 'val', 'test')}
    print(f'vòng {v}: {n}')

vòng 0: {'train': 2111, 'val': 201, 'test': 498}
vòng 1: {'train': 2241, 'val': 221, 'test': 294}
vòng 2: {'train': 1990, 'val': 203, 'test': 474}
vòng 3: {'train': 2176, 'val': 215, 'test': 354}
vòng 4: {'train': 1806, 'val': 162, 'test': 885}


In [3]:
# 3. Hàm train + chấm một lượt. Ghi JSON ngay khi xong; có JSON thì bỏ qua.
from ultralytics import YOLO
import numpy as np, torch

CAU_HINH = dict(epochs=150, patience=20, batch=16, imgsz=640, lr0=0.005, degrees=15,
                flipud=0.5, fliplr=0.5, mosaic=1.0, close_mosaic=15, optimizer='auto',
                workers=4, plots=False, verbose=False)
KIEN_TRUC = {           # tên: (yaml, kiểu hộp)
    'yolo26s-obb': ('yolo26s-obb.yaml', 'obb'),
    'yolov9c':     ('yolov9c.yaml',     'hbb'),
    'yolo26s':     ('yolo26s.yaml',     'hbb'),
    'yolov8s-obb': ('yolov8s-obb.yaml', 'obb'),
}

def theo_lop(b, thuoc):
    # b.p / b.r / b.ap50 chỉ có ở lớp CÓ mẫu trong test; ánh xạ về đủ 4 lớp qua ap_class_index, lớp vắng = None
    v = getattr(b, thuoc); ra = [None] * 4
    for k, ci in enumerate(b.ap_class_index):
        ra[int(ci)] = round(float(v[k]), 4)
    return ra

def cham_tai_conf(model, data_yaml, conf):
    # OBBMetrics lẫn DetMetrics đều để kết quả ở .box (ultralytics 8.4)
    r = model.val(data=data_yaml, split='test', conf=conf, iou=0.5, plots=False, verbose=False)
    b = r.box
    return dict(P_theo_lop=theo_lop(b, 'p'), R_theo_lop=theo_lop(b, 'r'), mAP50=round(float(b.map50), 4))

def mot_luot(ten, vong, hat):
    yaml_, kieu = KIEN_TRUC[ten]
    tag = f'{ten}_v{vong}_s{hat}'
    ra = f'{KQ}/{tag}.json'
    if os.path.exists(ra):
        print('  bỏ qua, đã có:', tag); return json.load(open(ra))
    goc = dung(vong, kieu); dy = f'{goc}/data.yaml'
    run = f'{KQ}/runs/{tag}'
    last = f'{run}/weights/last.pt'
    t0 = time.time()
    if os.path.exists(last):
        print('  nối tiếp từ last.pt:', tag)
        m = YOLO(last); m.train(resume=True)
    else:
        m = YOLO(yaml_)
        m.train(data=dy, seed=hat, project=f'{KQ}/runs', name=tag, exist_ok=True, **CAU_HINH)
    best = f'{run}/weights/best.pt'
    m = YOLO(best)
    # chấm test: mAP (conf mặc định 0,001) + P/R ở conf 0,25 và 0,5
    r = m.val(data=dy, split='test', plots=False, verbose=False)
    b = r.box
    kq = dict(kien_truc=ten, yaml=yaml_, kieu_hop=kieu, vong=vong, hat=hat,
              tham_so_M=round(sum(p.numel() for p in m.model.parameters()) / 1e6, 2),
              epoch_chay=int(len(open(f'{run}/results.csv').read().splitlines()) - 1),
              phut=round((time.time() - t0) / 60, 1),
              test=dict(mAP50=round(float(b.map50), 4), mAP50_95=round(float(b.map), 4),
                        AP50_theo_lop=theo_lop(b, 'ap50'), lop=LOP),
              conf_0_25=cham_tai_conf(m, dy, 0.25),
              conf_0_5=cham_tai_conf(m, dy, 0.5),
              cau_hinh=CAU_HINH)
    json.dump(kq, open(ra, 'w'), ensure_ascii=False, indent=1)
    print(f'  {tag}: mAP50 {kq["test"]["mAP50"]:.4f}  mAP50-95 {kq["test"]["mAP50_95"]:.4f}  '
          f'AP50 theo lớp {kq["test"]["AP50_theo_lop"]}  ({kq["phut"]} phút, {kq["epoch_chay"]} epoch)')
    return kq   # giữ best.pt trong runs/ để chấm lại sau

In [ ]:
# 4. BẢNG A — vòng 0, bốn kiến trúc. yolo26s-obb 3 hạt, còn lại 1 hạt trước.
# Thứ tự: pipeline mình trước, rồi kiến trúc Darabi, rồi hai đối chứng.
LICH_A = [('yolo26s-obb', 0, 0), ('yolov9c', 0, 0), ('yolo26s', 0, 0), ('yolov8s-obb', 0, 0),
          ('yolo26s-obb', 0, 1), ('yolo26s-obb', 0, 2),
          ('yolov9c', 0, 1), ('yolo26s', 0, 1), ('yolov8s-obb', 0, 1),
          ('yolov9c', 0, 2), ('yolo26s', 0, 2), ('yolov8s-obb', 0, 2)]
for ten, v, s in LICH_A:
    print(f'=== {ten}  vòng {v}  hạt {s} ===')
    mot_luot(ten, v, s)

=== yolo26s-obb  vòng 0  hạt 0 ===
  bỏ qua, đã có: yolo26s-obb_v0_s0
=== yolov9c  vòng 0  hạt 0 ===
  bỏ qua, đã có: yolov9c_v0_s0
=== yolo26s  vòng 0  hạt 0 ===
  bỏ qua, đã có: yolo26s_v0_s0
=== yolov8s-obb  vòng 0  hạt 0 ===
  bỏ qua, đã có: yolov8s-obb_v0_s0
=== yolo26s-obb  vòng 0  hạt 1 ===
  bỏ qua, đã có: yolo26s-obb_v0_s1
=== yolo26s-obb  vòng 0  hạt 2 ===
  bỏ qua, đã có: yolo26s-obb_v0_s2
=== yolov9c  vòng 0  hạt 1 ===
  bỏ qua, đã có: yolov9c_v0_s1
=== yolo26s  vòng 0  hạt 1 ===
  bỏ qua, đã có: yolo26s_v0_s1
=== yolov8s-obb  vòng 0  hạt 1 ===
  bỏ qua, đã có: yolov8s-obb_v0_s1
=== yolov9c  vòng 0  hạt 2 ===
  bỏ qua, đã có: yolov9c_v0_s2
=== yolo26s  vòng 0  hạt 2 ===
  nối tiếp từ last.pt: yolo26s_v0_s2
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close

In [4]:
# 5. B3 — yolo26s-obb, vòng 1..4 (vòng 0 đã có ở ô 4). Hạt 0 trước, đủ 5 vòng rồi mới thêm hạt.
LICH_B3 = [('yolo26s-obb', v, 0) for v in range(1, 5)] + [('yolo26s-obb', v, s) for s in (1, 2) for v in range(1, 5)]
for ten, v, s in LICH_B3:
    print(f'=== {ten}  vòng {v}  hạt {s} ===')
    mot_luot(ten, v, s)

=== yolo26s-obb  vòng 1  hạt 0 ===
  bỏ qua, đã có: yolo26s-obb_v1_s0
=== yolo26s-obb  vòng 2  hạt 0 ===
  bỏ qua, đã có: yolo26s-obb_v2_s0
=== yolo26s-obb  vòng 3  hạt 0 ===
  bỏ qua, đã có: yolo26s-obb_v3_s0
=== yolo26s-obb  vòng 4  hạt 0 ===
  bỏ qua, đã có: yolo26s-obb_v4_s0
=== yolo26s-obb  vòng 1  hạt 1 ===
  bỏ qua, đã có: yolo26s-obb_v1_s1
=== yolo26s-obb  vòng 2  hạt 1 ===
  bỏ qua, đã có: yolo26s-obb_v2_s1
=== yolo26s-obb  vòng 3  hạt 1 ===
  bỏ qua, đã có: yolo26s-obb_v3_s1
=== yolo26s-obb  vòng 4  hạt 1 ===
  bỏ qua, đã có: yolo26s-obb_v4_s1
=== yolo26s-obb  vòng 1  hạt 2 ===
  bỏ qua, đã có: yolo26s-obb_v1_s2
=== yolo26s-obb  vòng 2  hạt 2 ===
  bỏ qua, đã có: yolo26s-obb_v2_s2
=== yolo26s-obb  vòng 3  hạt 2 ===
  bỏ qua, đã có: yolo26s-obb_v3_s2
=== yolo26s-obb  vòng 4  hạt 2 ===
  nối tiếp từ last.pt: yolo26s-obb_v4_s2
Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False

In [5]:
# 6. Gom bảng — chạy bất cứ lúc nào, đọc mọi JSON đang có
import pandas as pd
fs = sorted(glob.glob(f'{KQ}/*.json'))
R = [json.load(open(f)) for f in fs]
df = pd.DataFrame([dict(kien_truc=r['kien_truc'], hop=r['kieu_hop'], vong=r['vong'], hat=r['hat'],
                        mAP50=r['test']['mAP50'], mAP50_95=r['test']['mAP50_95'],
                        R_0_25_Single=r['conf_0_25']['R_theo_lop'][0], R_0_25_Multi=r['conf_0_25']['R_theo_lop'][1],
                        R_0_25_Bypass=r['conf_0_25']['R_theo_lop'][2],
                        AP50_Single=r['test']['AP50_theo_lop'][0], AP50_Multi=r['test']['AP50_theo_lop'][1],
                        AP50_Bypass=r['test']['AP50_theo_lop'][2],
                        AP50_MOC=r['test']['AP50_theo_lop'][3],   # None ở vòng 3, 4 — đúng, không chấm riêng
                        phut=r['phut'], epoch=r['epoch_chay']) for r in R])
print('--- Bảng A: vòng 0, trung bình ± lệch theo hạt ---')
a = df[df.vong == 0].groupby(['kien_truc', 'hop']).agg(n_hat=('hat', 'count'), mAP50=('mAP50', 'mean'), sd=('mAP50', 'std'),
                                                       mAP50_95=('mAP50_95', 'mean'), AP50_Single=('AP50_Single', 'mean'),
                                                       AP50_Multi=('AP50_Multi', 'mean'), AP50_Bypass=('AP50_Bypass', 'mean'),
                                                       phut=('phut', 'mean')).round(4)
print(a.to_string())
print()
print('--- B3: yolo26s-obb, theo vòng (hạt 0) ---')
b = df[(df.kien_truc == 'yolo26s-obb') & (df.hat == 0)].sort_values('vong')[['vong', 'mAP50', 'mAP50_95', 'AP50_Single', 'AP50_Multi', 'AP50_Bypass', 'R_0_25_Single', 'R_0_25_Multi', 'R_0_25_Bypass']]
print(b.to_string(index=False))
if len(b) == 5: print('\ntrung bình 5 vòng: mAP50 %.4f ± %.4f' % (b.mAP50.mean(), b.mAP50.std()))
df.to_csv(f'{KQ}/tong_hop.csv', index=False); print('\n->', f'{KQ}/tong_hop.csv')

--- Bảng A: vòng 0, trung bình ± lệch theo hạt ---
                 n_hat   mAP50      sd  mAP50_95  AP50_Single  AP50_Multi  AP50_Bypass      phut
kien_truc   hop                                                                                 
yolo26s     hbb      3  0.6387  0.0056    0.5852       0.9178      0.6529       0.9831   58.8667
yolo26s-obb obb      3  0.6216  0.0236    0.6138       0.9076      0.6085       0.9703   65.4000
yolov8s-obb obb      3  0.6416  0.0074    0.6353       0.9307      0.6476       0.9881   70.9000
yolov9c     hbb      3  0.6475  0.0197    0.5861       0.9370      0.6571       0.9869  173.0000

--- B3: yolo26s-obb, theo vòng (hạt 0) ---
 vong  mAP50  mAP50_95  AP50_Single  AP50_Multi  AP50_Bypass  R_0_25_Single  R_0_25_Multi  R_0_25_Bypass
    0 0.6489    0.6421       0.9415      0.6630       0.9909         0.8923        0.6548         0.9663
    1 0.6723    0.6601       0.9262      0.7804       0.9826         0.9277        0.6250         0.9597
    2 0.